In [ ]:
# global
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scRNA
import scanpy as sc
import scvi

# constants
DATA_PATH = os.getcwd()

## Loading file 
here we load file, important is that we chave cells vs genes !!!

In [ ]:
filename = "learn_data/GSM5226574_C51ctr_raw_counts.csv.gz"
full_path = os.path.join(DATA_PATH, filename)
if not os.path.exists(full_path):
    raise FileNotFoundError(f"File {full_path} does not exist. Please check the path.")

adata = sc.read_csv(full_path).T # we want cells vs genes

## Doublet removal 
### Introduction
here we are removing cells which end up in same droplet (it comes from separation technique)

### Training SCVI model 
#### How does it work
TODO what is it excaclty 
#### Important need to be done on individual samples (why?) 
and than it will be predicting if there are doublets?

### Training SOLO model
#### How does it work
Why we training this based on the first model (???)

### Reducing data
#### Based on model
We can do it if it is not big percent of data (we decide about it)
#### Own metric 
we have numerical value assignment to how sure prediction is, and we have compare difference between this too, if model predicts that huge amount of data are doublets we can 

### Questions
- know-how: Why he removes last two digits ???
- model: how does it exactly works?? how the models predict that ??? 
- model training: if we have different datasets, we should train it on every data set separately or we can merge training data, so it will be trained better? how to choose here data for training and why? What model is it?
- reducing data: based on displot can we calculate the error of first type here ???

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", subset=True)

In [ ]:
scvi.model.SCVI.setup_anndata(adata)
vae = scvi.model.SCVI(adata)
vae.train()

In [ ]:
solo = scvi.external.SOLO.from_scvi_model(vae)
solo.train(datasplitter_kwargs={"num_workers": 4})





In [ ]:
# creating a dataframe with predictions
df = solo.predict()
df['prediction'] = solo.predict(soft=False)
# calculating difference between predictions
df['dif'] = df.doublet - df.singlet
# removing dash zeros
df.index = df.index.map(lambda x: x[:-2])
df

In [ ]:
# counting predictions
df.groupby('prediction').count()

In [ ]:
# distribution showing what can be omitted (the hyphotesis that those are TRUE dublets are low)
sns.displot(df[df.prediction == 'singlet'], x='dif')

In [ ]:
df[~df['dif'] > 0.2].count()

In [ ]:
# applying our decision 
doublets = df[(df.prediction == 'doublet') & (df.dif > 1)]
doublets

In [ ]:
# loading data again (?) 
adata = sc.read_csv(full_path).T 
# finding doublets in our observations (barcodes) (True values)
adata.obs['doublet'] = adata.obs.index.isin(doublets.index)
adata = adata[~adata.obs['doublet']]
adata

# Preprocessing the data

In [ ]:
# sprawdzić batch effekt

# sprawdzać w headach czy są nagłówki czy nie 

# atlasy do przejrzenia 

# wyciszenie genu -> wpływ na składanie trudniejszych białek (białka opiekuńcze) 
# nie posiadanie genu, wpływ na skutek 
# zasób wiedzy o ścieżkach ???? żeby to wykorzystać
# przejrzenie atlasów, i publiczne perturbation data sets. 

# trzeba zrobić perturbation data sets !!! (6) - 

# encodery autowariacyjne (to trzeba zrobić) 

# TODO 3 pierwsze dataset
# TODO koekspresja doczytane tego !!! (statystyka) 
# TODO czytanie o modelach - wszystkie istniejące modele (TO JEST WAŻNE) 

# zanotować do ścieżęk ten dataset !!! (analiza wzbogaceń genów) (KEG ???)

# batch effekt łączenia baz danych - doczytać o zrozumieć jak to działa

# gene set enrichment anazyylys - jak często się pojawia funkcja (?) jak to sprawdzić - zliczenie funkcji zamiast sprawdzanie czy są wygaszane. 

# analiza wzbogacenia genów - mamy liste genów któe są potencjalnie zaburzone i to wyszukuje w bazie danych procesów biologicznych - grupy genów a nie jeden gen !!!) 

# sygnalizacja komórkowa - nie tylko o białka się opieramy 